Amazon Berkely images (L2 real)

In [3]:
pip install "accelerate==0.33.0" --force-reinstall -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tables 3.8.0 requires blosc2~=2.0.0, which is not installed.
tables 3.8.0 requires cython>=0.29.21, which is not installed.
conda-repo-cli 1.0.41 requires requests_mock, which is not installed.
spyder 5.4.3 requires pyqt5<5.16, which is not installed.
spyder 5.4.3 requires pyqtwebengine<5.16, which is not installed.
gensim 4.3.0 requires FuzzyTM>=0.4.0, which is not installed.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.15.0 which is incompatible.
torchvision 0.16.2 requires torch==2.1.2, but you have torch 2.14.0 which is incompatible.
streamlit 1.39.0 requires packaging<25,>=20, but you have packaging 26.3 which is incompatible.
streamlit 1.39.0 requires pillow<11,>=7.1.0, but you have pillow 12.2.0 which is incompatible.
streamlit 1.39.0 requires protobuf<6,>=3.20, but you have protobuf

In [1]:
import os, time, torch
from diffusers import StableDiffusionPipeline
from PIL import Image

# ── Configuration ────────────────────────────────────────────
PROJECT_DIR = os.path.dirname(os.path.abspath('__file__'))
OUTPUT_DIR  = os.path.join(PROJECT_DIR, 'generated_fake')
MODEL_ID    = 'runwayml/stable-diffusion-v1-5'
SEED        = 42
N_STEPS     = 30
GUIDANCE    = 7.5
IMG_SIZE    = 512

# 17 texture + 17 fractal + 16 abstract = 50 images
PROMPTS = [
    # ── TEXTURE (17) ─────────────────────────────────────────
    ('texture', 'close-up macro photograph of rough stone texture, detailed surface, photorealistic'),
    ('texture', 'seamless fabric weave texture, fine linen threads, macro photography'),
    ('texture', 'cracked dry earth texture, deep crevices, top-down aerial view'),
    ('texture', 'wood grain texture, aged oak, fine detail, photorealistic'),
    ('texture', 'rusted metal surface texture, oxidized iron, macro detail'),
    ('texture', 'smooth marble texture, white and grey veins, polished surface'),
    ('texture', 'leather texture, brown hide, fine grain, close-up macro'),
    ('texture', 'concrete wall texture, grey cement, rough surface, close-up'),
    ('texture', 'water surface ripple texture, light reflections, abstract top view'),
    ('texture', 'crystalline ice texture, frozen patterns, translucent blue'),
    ('texture', 'sand dune texture, fine grains, wind patterns, desert'),
    ('texture', 'bark texture, old tree, deep furrows, macro photography'),
    ('texture', 'snake skin texture, scales pattern, close-up macro detail'),
    ('texture', 'woven basket texture, rattan pattern, overhead view'),
    ('texture', 'shattered glass texture, transparent cracks, dark background'),
    ('texture', 'cobblestone pavement texture, mossy gaps, top-down view'),
    ('texture', 'wool felt texture, soft fibres, colorful, microscopic detail'),

    # ── FRACTAL (17) ─────────────────────────────────────────
    ('fractal', 'mandelbrot set fractal art, vibrant electric colors, ultra detailed, infinite zoom'),
    ('fractal', 'julia set fractal art, electric blue and purple, glowing edges, dark background'),
    ('fractal', 'sierpinski triangle fractal, geometric recursion, neon colors, black background'),
    ('fractal', 'burning ship fractal, fiery orange and red, abstract digital art'),
    ('fractal', 'nova fractal, spiraling galaxy pattern, cosmic deep space aesthetic'),
    ('fractal', 'lyapunov fractal, green and red islands, mathematical digital art'),
    ('fractal', 'fractal tree branching structure, golden ratio, autumn colors, symmetry'),
    ('fractal', 'mandelbulb 3d fractal, photorealistic render, iridescent metallic surface'),
    ('fractal', 'coral fractal growth pattern, underwater organic shapes, cyan light'),
    ('fractal', 'newton fractal convergence basin, colorful zones, complex mathematical plane'),
    ('fractal', 'fractal flame render, smoke-like tendrils, deep black background, colorful'),
    ('fractal', 'menger sponge 3d fractal, recursive cube structure, grey and white'),
    ('fractal', 'romanesco broccoli fractal, natural logarithmic spiral, vivid green'),
    ('fractal', 'fractal coastline, recursive jagged edges, aerial ocean view, blue'),
    ('fractal', 'barnsley fern fractal, green botanical recursion, black background'),
    ('fractal', 'apollonian gasket fractal, packed circles, rainbow gradient'),
    ('fractal', 'dragon curve fractal, intricate iteration pattern, neon on dark'),

    # ── ABSTRACT ART (16) ────────────────────────────────────
    ('abstract', 'abstract oil painting, bold expressive brushstrokes, vibrant impasto texture'),
    ('abstract', 'abstract watercolor wash, flowing colors blending, soft dreamy edges'),
    ('abstract', 'geometric abstract art, mondrian style, primary colors, bold black grid'),
    ('abstract', 'abstract expressionism drip painting, jackson pollock inspired, colorful chaos'),
    ('abstract', 'abstract fluid art pour, swirling acrylic, marble effect, vibrant'),
    ('abstract', 'minimal abstract color field painting, rothko style, warm gradients'),
    ('abstract', 'abstract digital glitch art, pixel corruption, neon colors, scanlines'),
    ('abstract', 'abstract neural network visualization, glowing nodes and connections'),
    ('abstract', 'abstract long exposure light painting, colorful light streaks, dark'),
    ('abstract', 'abstract cubist composition, fragmented geometric forms, earth tones'),
    ('abstract', 'abstract ink blot symmetrical rorschach pattern, black on white'),
    ('abstract', 'abstract smooth gradient mesh, color transitions, soft pastel palette'),
    ('abstract', 'abstract stained glass mosaic, vibrant colored fragments, dark lead outlines'),
    ('abstract', 'abstract generative algorithmic art, parametric patterns, cool blue'),
    ('abstract', 'coloured smoke art, swirling plumes on black background, vivid'),
    ('abstract', 'abstract low-poly geometric landscape, triangulated facets, sunset colors'),
]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Device ───────────────────────────────────────────────────
DEVICE = 'mps' if torch.backends.mps.is_available() else \
         'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# ── Load pipeline ─────────────────────────────────────────────
print(f'Loading {MODEL_ID} … (first run downloads ~4 GB)')
dtype = torch.float16 if DEVICE in ('cuda', 'mps') else torch.float32

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False,
    low_cpu_mem_usage=True,
)
pipe = pipe.to(DEVICE)
pipe.set_progress_bar_config(disable=True)
if DEVICE == 'mps':
    pipe.enable_attention_slicing()  # saves MPS memory

# ── Generate ──────────────────────────────────────────────────
total = len(PROMPTS)
print(f'Generating {total} images …\n')

for i, (category, prompt) in enumerate(PROMPTS, 1):
    fname     = f'{category}_{i:03d}.png'
    dest_path = os.path.join(OUTPUT_DIR, fname)

    if os.path.exists(dest_path):
        print(f'  [{i:02d}/{total}] already exists – {fname}')
        continue

    # Note: generator on CPU avoids torch 2.1 MPS generator bug
    generator = torch.Generator(device='cpu').manual_seed(SEED + i)
    result    = pipe(
        prompt,
        num_inference_steps=N_STEPS,
        guidance_scale=GUIDANCE,
        height=IMG_SIZE, width=IMG_SIZE,
        generator=generator,
    )
    result.images[0].save(dest_path)
    print(f'  [{i:02d}/{total}] ✓ [{category}]  {fname}')
    print(f'         » {prompt[:72]}…')

print(f'\n══ Done!  {total} images → {OUTPUT_DIR}')



/Users/andrea/anaconda3/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


RuntimeError: Failed to import diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion because of the following error (look up to see its traceback):
Failed to import diffusers.loaders.ip_adapter because of the following error (look up to see its traceback):
cannot import name 'SiglipImageProcessor' from 'transformers' (/Users/andrea/anaconda3/lib/python3.11/site-packages/transformers/__init__.py)

In [ ]:
# ============================================================
# Generate 50 fake images (texture / fractal / abstract art)
# Pure numpy + PIL — no ML libraries, no GPU, runs in ~30s
# 17 textures  (001-017)
# 17 fractals  (018-034)
# 16 abstract  (035-050)
# ============================================================
import os, colorsys
import numpy as np
from PIL import Image, ImageDraw, ImageFilter
import matplotlib.cm as cm

PROJECT_DIR = os.path.dirname(os.path.abspath('__file__'))
OUTPUT_DIR  = os.path.join(PROJECT_DIR, 'generated_fake')
SIZE        = 512
SEED        = 42
os.makedirs(OUTPUT_DIR, exist_ok=True)
rng  = np.random.default_rng(SEED)
saved = 0

# ── Helpers ──────────────────────────────────────────────────
def save(data, name):
    global saved
    if isinstance(data, np.ndarray):
        img = Image.fromarray(data.astype(np.uint8))
    else:
        img = data
    img.save(os.path.join(OUTPUT_DIR, name))
    saved += 1
    print(f'  [{saved:02d}/50] ✓ {name}')

def cmap(vals, name='inferno'):
    v = np.clip(vals, 0, 1)
    _cm = cm.colormaps[name] if hasattr(cm, 'colormaps') else cm.get_cmap(name)
    return (_cm(v)[:, :, :3] * 255).astype(np.uint8)


def fnoise(octaves=6, base=4, size=SIZE):
    """Multi-octave fractal noise via bilinear upsampling."""
    out = np.zeros((size, size))
    amp = 1.0
    for o in range(octaves):
        gs = max(2, base * (2 ** o))
        g  = rng.random((gs + 1, gs + 1))
        up = np.array(Image.fromarray((g * 255).astype(np.uint8)).resize((size, size), Image.BILINEAR)) / 255.0
        out += up * amp
        amp *= 0.5
    out -= out.min(); out /= out.max() + 1e-8
    return out

def mandelbrot(x0, x1, y0, y1, iters=200):
    x = np.linspace(x0, x1, SIZE); y = np.linspace(y0, y1, SIZE)
    C = x[None, :] + 1j * y[:, None]
    Z = np.zeros_like(C); M = np.zeros((SIZE, SIZE), float)
    for _ in range(iters):
        mask = np.abs(Z) <= 2
        Z[mask] = Z[mask] ** 2 + C[mask]; M += mask
    return M / iters

def julia(c, scale=2.0, iters=200, power=2):
    x = np.linspace(-scale, scale, SIZE); y = np.linspace(-scale, scale, SIZE)
    Z = x[None, :] + 1j * y[:, None]; M = np.zeros((SIZE, SIZE), float)
    for _ in range(iters):
        mask = np.abs(Z) <= 2
        Z[mask] = Z[mask] ** power + c; M += mask
    return M / iters

S = SIZE
x_ = np.linspace(0, 4 * np.pi, S); y_ = np.linspace(0, 4 * np.pi, S)
X, Y = np.meshgrid(x_, y_)
XI, YI = np.meshgrid(np.linspace(-np.pi, np.pi, S), np.linspace(-np.pi, np.pi, S))

# ── TEXTURES ────────────────────────────────────────────────
print('\n── TEXTURES ──')

# 1 Marble
n = fnoise(6, 4)
save(cmap(np.sin(X + n * 8) * 0.5 + 0.5, 'gray'),       'texture_001_marble.png')
# 2 Wood
r_ = np.sqrt((XI) ** 2 + (YI) ** 2)
save(cmap(np.sin(r_ * 20 + fnoise() * 4) * 0.5 + 0.5, 'YlOrBr'), 'texture_002_wood.png')
# 3 Clouds
save(cmap(fnoise(8, 2), 'gray'),                          'texture_003_clouds.png')
# 4 Lava
n2 = fnoise(6, 3)
lv = (np.sin(Y * 3 + n2 * 8) * 0.5 + 0.5) * n2
save(cmap(lv / lv.max(), 'hot'),                          'texture_004_lava.png')
# 5 Water ripples
rp = np.sin(np.sqrt(XI**2 + YI**2) * 18) * 0.5 + fnoise(4, 8) * 0.3 + 0.2
save(cmap(rp / rp.max(), 'ocean'),                        'texture_005_ripples.png')
# 6 Rust
n3 = fnoise(7, 3)
save(cmap(n3 * (1 - fnoise(4, 6) * 0.4), 'copper'),      'texture_006_rust.png')
# 7 Granite
n4 = fnoise(5, 4)
dots = rng.random((S, S)) > 0.97
blur_dots = np.array(Image.fromarray((dots * 220).astype(np.uint8)).filter(ImageFilter.GaussianBlur(1.5))) / 255.0
save(cmap(n4 * 0.7 + blur_dots * 0.3, 'gray'),            'texture_007_granite.png')
# 8 Ice
n5 = fnoise(7, 2)
ice = n5 + np.abs(np.sin(X * 12)) * 0.15 + np.abs(np.sin(Y * 12)) * 0.15
save(cmap(ice / ice.max(), 'cool'),                        'texture_008_ice.png')
# 9 Sand
save(cmap(fnoise(4, 8) * 0.6 + fnoise(6, 4) * 0.4, 'YlOrBr'), 'texture_009_sand.png')
# 10 Moss
n6 = fnoise(6, 3)
moss = n6 + np.abs(np.sin(X * 6 + n6 * 3)) * 0.2
save(cmap(moss / moss.max(), 'YlGn'),                     'texture_010_moss.png')
# 11 Scales (Voronoi distance field)
pts = rng.random((25, 2)) * S
gy, gx = np.mgrid[0:S, 0:S]
dists = np.stack([np.sqrt((gx - p[0])**2 + (gy - p[1])**2) for p in pts])
d1 = np.sort(dists, axis=0)[0]; d2 = np.sort(dists, axis=0)[1]
scales_v = (d2 - d1) / (d2 + 1e-8)
save(cmap(scales_v / scales_v.max(), 'YlOrBr'),           'texture_011_scales.png')
# 12 Cracked earth
crack = 1 - np.clip((d2 - d1) * 0.25, 0, 1)
save(cmap(fnoise(5, 3) * 0.5 + crack * 0.5, 'hot'),      'texture_012_cracks.png')
# 13 Woven fabric
wx = np.abs(np.sin(X * 25)); wy = np.abs(np.sin(Y * 25 + 0.5))
save(cmap(np.where(wx > wy, wx, wy * 0.7), 'copper'),     'texture_013_fabric.png')
# 14 Carbon fiber
cf = np.abs(np.sin(X * 40)) * np.abs(np.cos(Y * 4)) + np.abs(np.sin(Y * 40)) * np.abs(np.cos(X * 4))
save(cmap(cf / cf.max(), 'gray'),                          'texture_014_carbon.png')
# 15 Nebula (RGB noise layers)
neb = np.stack([(fnoise(8, 2) * 255).astype(np.uint8),
                (fnoise(6, 3) * 128).astype(np.uint8),
                (fnoise(4, 4) * 255).astype(np.uint8)], axis=-1)
save(neb,                                                   'texture_015_nebula.png')
# 16 Brushed metal
metal = np.zeros((S, S))
for i in range(0, S, 2): metal[i, :] = fnoise(2, 32)[i, :]
metal = (metal - metal.min()) / (metal.max() - metal.min() + 1e-8)
metal_rgb = np.stack([metal * 180 + 40, metal * 190 + 45, metal * 210 + 50], axis=-1)
save(metal_rgb.astype(np.uint8),                           'texture_016_metal.png')
# 17 Coral/organic
n11 = fnoise(7, 2)
save(cmap(np.sin(n11 * 18) * 0.5 + 0.5, 'pink'),         'texture_017_coral.png')

# ── FRACTALS ────────────────────────────────────────────────
print('\n── FRACTALS ──')

# 18-22 Mandelbrot at different zoom windows
for idx, (x0, x1, y0, y1, cm_) in enumerate([
    (-2.5,  1.0,  -1.25, 1.25, 'inferno'),
    (-0.80, -0.60,  0.10, 0.30, 'plasma'),
    (-0.15, -0.05,  0.95, 1.05, 'viridis'),
    (-1.80, -1.60, -0.10, 0.10, 'magma'),
    (-0.75, -0.65,  0.00, 0.10, 'twilight'),
], 1):
    save(cmap(mandelbrot(x0, x1, y0, y1), cm_), f'fractal_{17+idx:03d}_mandelbrot.png')

# 23-28 Julia sets with classic parameters
for idx, (c, cm_, sc) in enumerate([
    (-0.70 + 0.270j, 'inferno', 2.0),
    (-0.40 + 0.600j, 'plasma',  2.0),
    ( 0.285 + 0.010j,'viridis', 2.0),
    (-0.835 - 0.232j, 'magma',  2.0),
    (-0.727 + 0.189j,'rainbow', 2.0),
    ( 0.000 + 0.800j, 'cool',   2.0),
], 1):
    save(cmap(julia(c, scale=sc), cm_), f'fractal_{22+idx:03d}_julia.png')

# 29 Burning Ship
xb = np.linspace(-2.5, 1.5, S); yb = np.linspace(-2.0, 0.5, S)
C_ = xb[None, :] + 1j * yb[:, None]
Z_ = np.zeros_like(C_); M_ = np.zeros((S, S), float)
for _ in range(200):
    mask_ = np.abs(Z_) <= 2
    Z_[mask_] = (np.abs(Z_[mask_].real) + 1j * np.abs(Z_[mask_].imag)) ** 2 + C_[mask_]
    M_ += mask_
save(cmap(M_ / 200, 'hot'),                                'fractal_029_burning_ship.png')

# 30 Newton fractal (z^3 = 1)
xn = np.linspace(-2, 2, S); yn = np.linspace(-2, 2, S)
Zn = xn[None, :] + 1j * yn[:, None]
roots_n = [1+0j, -0.5+0.866j, -0.5-0.866j]
for _ in range(60): Zn = Zn - (Zn**3 - 1) / (3 * Zn**2 + 1e-10)
newton_img = np.zeros((S, S, 3), dtype=np.uint8)
cols_n = [(220, 50, 50), (50, 200, 80), (50, 80, 220)]
for ri, root in enumerate(roots_n):
    which = np.abs(Zn - root) == np.min([np.abs(Zn - r) for r in roots_n], axis=0)
    for ch in range(3): newton_img[:, :, ch][which] = cols_n[ri][ch]
save(newton_img,                                           'fractal_030_newton.png')

# 31 Mandelbrot smooth (escape-time smooth)
xs = np.linspace(-2.1, 0.6, S); ys = np.linspace(-1.3, 1.3, S)
Cs = xs[None, :] + 1j * ys[:, None]
Zs = np.zeros_like(Cs); Ms = np.zeros((S, S), float); AbsZ = np.zeros((S, S), float)
for _ in range(256):
    m = np.abs(Zs) <= 2
    Zs[m] = Zs[m] ** 2 + Cs[m]; Ms[m] += 1; AbsZ[m] = np.abs(Zs[m])
smooth = Ms - np.log2(np.log2(np.maximum(AbsZ, 1.01) + 1e-10))
smooth = np.clip((smooth - smooth.min()) / (smooth.max() - smooth.min() + 1e-8), 0, 1)
save(cmap(smooth, 'hsv'),                                  'fractal_031_smooth.png')

# 32 Multibrot z^3
xm = np.linspace(-1.5, 1.5, S); ym = np.linspace(-1.5, 1.5, S)
Cm = xm[None, :] + 1j * ym[:, None]
Zm = np.zeros_like(Cm); Mm = np.zeros((S, S), float)
for _ in range(200):
    mk = np.abs(Zm) <= 2; Zm[mk] = Zm[mk]**3 + Cm[mk]; Mm += mk
save(cmap(Mm / 200, 'plasma'),                             'fractal_032_multibrot3.png')

# 33 Julia z^3
save(cmap(julia(0.4 + 0.2j, power=3), 'twilight'),        'fractal_033_julia_cubic.png')

# 34 Barnsley Fern (IFS)
fern = np.zeros((S, S), dtype=np.float32)
xf, yf = 0.0, 0.0
for _ in range(600_000):
    rv = rng.random()
    if   rv < 0.01: xf, yf = 0, 0.16 * yf
    elif rv < 0.86: xf, yf = 0.85*xf + 0.04*yf, -0.04*xf + 0.85*yf + 1.6
    elif rv < 0.93: xf, yf = 0.2*xf - 0.26*yf,   0.23*xf + 0.22*yf + 1.6
    else:           xf, yf = -0.15*xf + 0.28*yf,  0.26*xf + 0.24*yf + 0.44
    px = int((xf + 3) / 6 * (S - 1)); py = int((10 - yf) / 10 * (S - 1))
    if 0 <= px < S and 0 <= py < S: fern[py, px] = min(1.0, fern[py, px] + 0.05)
fern_blur = np.array(Image.fromarray((fern * 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(0.5)))
fern_rgb = np.zeros((S, S, 3), np.uint8); fern_rgb[:, :, 1] = fern_blur
save(fern_rgb,                                             'fractal_034_fern.png')

# ── ABSTRACT ────────────────────────────────────────────────
print('\n── ABSTRACT ──')

# 35 Wave interference
w1 = np.sin(XI * 5 + YI * 3); w2 = np.sin(XI * 3 - YI * 5)
save(cmap((w1 + w2 + 2) / 4, 'hsv'),                      'abstract_035_interference.png')

# 36 Color wheel
R_ = np.sqrt(XI**2 + YI**2) / np.pi
theta_ = np.arctan2(YI, XI)
h_ = (theta_ + np.pi) / (2 * np.pi)
s_ = np.clip(R_, 0, 1)
v_ = np.ones_like(h_)
h6_ = h_ * 6; i6_ = h6_.astype(int) % 6; f6_ = h6_ - np.floor(h6_)
p_ = v_ * (1 - s_); q_ = v_ * (1 - s_ * f6_); t_ = v_ * (1 - s_ * (1 - f6_))
rr = np.select([i6_==0,i6_==1,i6_==2,i6_==3,i6_==4],[v_,q_,p_,p_,t_],v_)
gg = np.select([i6_==0,i6_==1,i6_==2,i6_==3,i6_==4],[t_,v_,v_,q_,p_],p_)
bb = np.select([i6_==0,i6_==1,i6_==2,i6_==3,i6_==4],[p_,p_,t_,v_,v_],q_)
save(np.stack([(rr*255).astype(np.uint8),(gg*255).astype(np.uint8),(bb*255).astype(np.uint8)],-1), 'abstract_036_colorwheel.png')

# 37 Spirograph
img37 = Image.new('RGB', (S, S), (8, 8, 18))
d37 = ImageDraw.Draw(img37)
for R2, r2, d2 in [(200,127,80),(180,76,120),(160,51,100)]:
    for k in range(1000):
        for ti, t2 in [(k/1000*2*np.pi, (k+1)/1000*2*np.pi)]:
            x1a = S//2 + (R2-r2)*np.cos(ti) + d2*np.cos((R2-r2)/r2*ti)
            y1a = S//2 + (R2-r2)*np.sin(ti) - d2*np.sin((R2-r2)/r2*ti)
            x2a = S//2 + (R2-r2)*np.cos(t2) + d2*np.cos((R2-r2)/r2*t2)
            y2a = S//2 + (R2-r2)*np.sin(t2) - d2*np.sin((R2-r2)/r2*t2)
            col37 = tuple(int(c*255) for c in colorsys.hsv_to_rgb(k/1000, 1, 1))
            d37.line([(x1a, y1a), (x2a, y2a)], fill=col37, width=1)
save(img37,                                                'abstract_037_spirograph.png')

# 38 Lissajous
img38 = Image.new('RGB', (S, S), (5, 5, 15))
d38 = ImageDraw.Draw(img38)
for a38, b38, delta38 in [(3,4,np.pi/4),(5,6,np.pi/3),(7,8,np.pi/6)]:
    ts = np.linspace(0, 2*np.pi, 4000)
    xs = (np.sin(a38*ts + delta38)*0.45 + 0.5) * S
    ys = (np.sin(b38*ts)*0.45 + 0.5) * S
    pts_l = [(float(xs[i]), float(ys[i])) for i in range(len(ts))]
    col38 = tuple(int(c*255) for c in colorsys.hsv_to_rgb(a38/10, 0.9, 1))
    d38.line(pts_l, fill=col38, width=1)
save(img38,                                                'abstract_038_lissajous.png')

# 39 Polar rose
img39 = Image.new('RGB', (S, S), (10, 5, 20))
d39 = ImageDraw.Draw(img39)
for k39 in [3, 5, 7]:
    ts = np.linspace(0, 2*np.pi, 5000)
    rs = np.abs(np.cos(k39 * ts))
    xs = (rs * np.cos(ts) * 0.45 + 0.5) * S
    ys = (rs * np.sin(ts) * 0.45 + 0.5) * S
    pts39 = [(float(xs[i]), float(ys[i])) for i in range(len(ts))]
    col39 = tuple(int(c*255) for c in colorsys.hsv_to_rgb(k39/12, 1, 1))
    d39.line(pts39, fill=col39, width=2)
save(img39,                                                'abstract_039_polar_rose.png')

# 40 Voronoi color
pts40 = rng.random((30, 2)) * S
cols40 = (rng.random((30, 3)) * 255).astype(np.uint8)
gy40, gx40 = np.mgrid[0:S, 0:S]
ds40 = np.stack([np.sqrt((gx40-p[0])**2+(gy40-p[1])**2) for p in pts40])
closest40 = np.argmin(ds40, axis=0)
vor_img = cols40[closest40]
save(vor_img,                                              'abstract_040_voronoi.png')

# 41 Gradient circles
gc = np.zeros((S, S, 3), dtype=np.uint8)
for _ in range(12):
    cx, cy = rng.integers(50, S-50, size=2)
    rad = rng.integers(40, 150)
    hue41 = rng.random()
    gy41, gx41 = np.mgrid[0:S, 0:S]
    d41 = np.sqrt((gx41-cx)**2 + (gy41-cy)**2)
    alpha = np.clip(1 - d41/rad, 0, 1)
    col41 = np.array([int(c*255) for c in colorsys.hsv_to_rgb(hue41, 1, 1)], dtype=np.uint8)
    for ch in range(3): gc[:,:,ch] = np.clip(gc[:,:,ch].astype(int) + (alpha * col41[ch]).astype(int), 0, 255).astype(np.uint8)
save(gc,                                                   'abstract_041_gradient_circles.png')

# 42 Mondrian
img42 = Image.new('RGB', (S, S), (240, 235, 220))
d42 = ImageDraw.Draw(img42)
cols42 = [(220,40,40),(40,80,180),(240,200,40),(240,235,220),(30,30,30)]
rects42 = [(0,0,180,180),(0,182,180,400),(182,0,400,260),(182,262,400,400),(402,0,512,512)]
for r42, c42 in zip(rects42, cols42[:4]+cols42):
    d42.rectangle(r42, fill=c42)
for x42 in [180,400]: d42.line([(x42,0),(x42,S)], fill=(20,20,20), width=5)
for y42 in [180,260]: d42.line([(0,y42),(S,y42)], fill=(20,20,20), width=5)
save(img42,                                                'abstract_042_mondrian.png')

# 43 Kaleidoscope from noise
n_k = fnoise(6, 3)
h_k = (n_k * 6).astype(int) % 6; f_k = n_k * 6 - np.floor(n_k * 6)
kal_r = np.select([h_k==0,h_k==1,h_k==2,h_k==3,h_k==4],[1,1-f_k,0,0,f_k],1)
kal_g = np.select([h_k==0,h_k==1,h_k==2,h_k==3,h_k==4],[f_k,1,1,1-f_k,0],0)
kal_b = np.select([h_k==0,h_k==1,h_k==2,h_k==3,h_k==4],[0,0,f_k,1,1],1-f_k)
kal = np.stack([(kal_r*255).astype(np.uint8),(kal_g*255).astype(np.uint8),(kal_b*255).astype(np.uint8)],-1)
img_k = Image.fromarray(kal)
q = img_k.crop((0,0,S//2,S//2))
full_k = Image.new('RGB', (S,S))
full_k.paste(q, (0,0)); full_k.paste(q.transpose(Image.FLIP_LEFT_RIGHT), (S//2,0))
full_k.paste(q.transpose(Image.FLIP_TOP_BOTTOM), (0,S//2))
full_k.paste(q.transpose(Image.ROTATE_180), (S//2,S//2))
save(full_k,                                               'abstract_043_kaleidoscope.png')

# 44 Flow field (noise-guided)
img44 = Image.new('RGB', (S, S), (5, 5, 10))
d44 = ImageDraw.Draw(img44)
angle_field = fnoise(4, 2) * 2 * np.pi
for _ in range(300):
    x44, y44 = float(rng.integers(0, S)), float(rng.integers(0, S))
    pts44 = [(x44, y44)]
    hue44 = rng.random()
    for __ in range(40):
        xi44 = min(S-1, max(0, int(x44))); yi44 = min(S-1, max(0, int(y44)))
        ang44 = angle_field[yi44, xi44]
        x44 += np.cos(ang44) * 3; y44 += np.sin(ang44) * 3
        if not (0 <= x44 < S and 0 <= y44 < S): break
        pts44.append((x44, y44))
    if len(pts44) > 1:
        col44 = tuple(int(c*255) for c in colorsys.hsv_to_rgb(hue44, 0.9, 0.9))
        d44.line(pts44, fill=col44, width=1)
save(img44,                                                'abstract_044_flow_field.png')

# 45 Plasma
plasma = (np.sin(X) + np.sin(Y) + np.sin((X+Y)/2) + np.sin(np.sqrt(XI**2+YI**2)*3)) / 4
save(cmap((plasma - plasma.min())/(plasma.max()-plasma.min()), 'hsv'), 'abstract_045_plasma.png')

# 46 Concentric rings
R46 = np.sqrt(XI**2 + YI**2)
rings = np.sin(R46 * 12) * 0.5 + fnoise(3, 16) * 0.3 + 0.2
save(cmap(rings/rings.max(), 'rainbow'), 'abstract_046_rings.png')

# 47 Smoke / fluid
smoke = fnoise(8, 1)
smoke_c = np.stack([
    cmap(smoke, 'Reds')[:,:,0],
    cmap(fnoise(8,2), 'Blues')[:,:,2],
    cmap(fnoise(8,3), 'Greens')[:,:,1],
], axis=-1)
save(smoke_c, 'abstract_047_smoke.png')

# 48 Recursive squares
img48 = Image.new('RGB', (S,S), (10,10,15))
d48 = ImageDraw.Draw(img48)
for level in range(8):
    margin = 10 + level * 28
    hue48 = level / 8
    col48 = tuple(int(c*255) for c in colorsys.hsv_to_rgb(hue48, 0.9, 0.9))
    d48.rectangle([margin, margin, S-margin, S-margin], outline=col48, width=3)
save(img48, 'abstract_048_recursive_squares.png')

# 49 Abstract noise art (multi-layer HSV)
h49 = fnoise(6, 3); s49 = fnoise(4, 5); v49 = fnoise(5, 4)
h6_49 = h49 * 6; i49 = h6_49.astype(int) % 6; f49 = h6_49 - np.floor(h6_49)
p49 = v49*(1-s49); q49 = v49*(1-s49*f49); t49 = v49*(1-s49*(1-f49))
r49 = np.select([i49==0,i49==1,i49==2,i49==3,i49==4],[v49,q49,p49,p49,t49],v49)
g49 = np.select([i49==0,i49==1,i49==2,i49==3,i49==4],[t49,v49,v49,q49,p49],p49)
b49 = np.select([i49==0,i49==1,i49==2,i49==3,i49==4],[p49,p49,t49,v49,v49],q49)
save(np.stack([(r49*255).astype(np.uint8),(g49*255).astype(np.uint8),(b49*255).astype(np.uint8)],-1), 'abstract_049_noise_art.png')

# 50 Glitch art
base50 = cmap(fnoise(5, 3), 'hsv')
glitch = base50.copy()
for _ in range(30):
    y50 = rng.integers(0, S)
    h50 = rng.integers(1, 15)
    shift50 = rng.integers(-60, 60)
    glitch[y50:y50+h50, :] = np.roll(glitch[y50:y50+h50, :], shift50, axis=1)
    ch50 = rng.integers(0, 3)
    glitch[y50:y50+h50, :, ch50] = np.clip(glitch[y50:y50+h50, :, ch50].astype(int) + rng.integers(30,80), 0, 255)
save(glitch, 'abstract_050_glitch.png')

print(f'\n══ Done! {saved} images saved to: {OUTPUT_DIR}')




── TEXTURES ──


AttributeError: module 'matplotlib.cm' has no attribute 'colormaps'

In [4]:
# ============================================================
# Download ~45 real product images from the
# Amazon Berkeley Objects (ABO) dataset  (CC BY 4.0)
# https://amazon-berkeley-objects.s3.amazonaws.com/index.html
# ============================================================
import io, gzip, csv, os, random, time
import urllib.request

# ── Configuration ────────────────────────────────────────────
OUTPUT_DIR   = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                            'images')          # saved next to this notebook
N_IMAGES     = 45                              # how many images to download
RANDOM_SEED  = 42
DELAY_S      = 0.15                            # polite delay between requests

# ABO public S3 base URLs (no auth required, CC BY 4.0)
META_URL   = ('https://amazon-berkeley-objects.s3.amazonaws.com'
              '/images/metadata/images.csv.gz')
# after (correct)
IMAGE_BASE = 'https://amazon-berkeley-objects.s3.amazonaws.com/images/small/'


os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Step 1: download & parse the image metadata CSV ─────────
print('Downloading image metadata CSV …', flush=True)
with urllib.request.urlopen(META_URL, timeout=60) as resp:
    raw = resp.read()

with gzip.open(io.BytesIO(raw), 'rt', encoding='utf-8') as gz:
    reader   = csv.DictReader(gz)
    all_rows = list(reader)

print(f'  → {len(all_rows):,} images in the metadata')

# ── Step 2: pick N_IMAGES random entries ────────────────────
rng    = random.Random(RANDOM_SEED)
sample = rng.sample(all_rows, N_IMAGES)

# ── Step 3: download each image ─────────────────────────────
downloaded, skipped = 0, 0
for i, row in enumerate(sample, 1):
    path = row.get('path', '').lstrip('/')
    if not path:
        skipped += 1
        continue

    filename  = os.path.basename(path)
    dest_path = os.path.join(OUTPUT_DIR, filename)

    if os.path.exists(dest_path):
        print(f'  [{i:02d}/{N_IMAGES}] already exists – {filename}')
        downloaded += 1
        continue

    url = IMAGE_BASE + path
    try:
        with urllib.request.urlopen(url, timeout=30) as r:
            data = r.read()
        with open(dest_path, 'wb') as f:
            f.write(data)
        kb = len(data) / 1024
        print(f'  [{i:02d}/{N_IMAGES}] ✓ {filename}  ({kb:.1f} KB)')
        downloaded += 1
    except Exception as e:
        print(f'  [{i:02d}/{N_IMAGES}] ✗ {filename}  ERROR: {e}')
        skipped += 1

    time.sleep(DELAY_S)

print(f'\nDone!  {downloaded} images saved to: {OUTPUT_DIR}')
if skipped:
    print(f'       {skipped} entries skipped (missing path or download error)')


  → 398,212 images in the metadata
  [01/45] ✓ 9f4e2aa1.jpg  (7.0 KB)
  [02/45] ✓ 2a731e2f.jpg  (2.7 KB)
  [03/45] ✓ b963ae32.jpg  (3.3 KB)
  [04/45] ✓ e0f6448a.jpg  (9.6 KB)
  [05/45] ✓ 81d76f54.jpg  (10.2 KB)
  [06/45] ✓ 29672be0.jpg  (8.3 KB)
  [07/45] ✓ 06d2b41e.jpg  (4.7 KB)
  [08/45] ✓ 4ef433a0.jpg  (11.5 KB)
  [09/45] ✓ 01e5740b.jpg  (37.1 KB)
  [10/45] ✓ 7546b814.jpg  (6.6 KB)
  [11/45] ✓ 060cfb81.jpg  (10.4 KB)
  [12/45] ✓ 48973cac.jpg  (31.3 KB)
  [13/45] ✓ 5ddec866.jpg  (5.2 KB)
  [14/45] ✓ 1f9eff69.jpg  (7.3 KB)
  [15/45] ✓ e7e90dbc.jpg  (9.0 KB)
  [16/45] ✓ 5b338554.jpg  (6.6 KB)
  [17/45] ✓ 1ce4d1f5.jpg  (2.8 KB)
  [18/45] ✓ 6f415cf3.jpg  (4.5 KB)
  [19/45] ✓ b6498bad.jpg  (10.0 KB)
  [20/45] ✓ 4ce31136.jpg  (5.6 KB)
  [21/45] ✓ 5810c13d.jpg  (4.9 KB)
  [22/45] ✓ 8da1ea8e.jpg  (13.5 KB)
  [23/45] ✓ 60d4680f.jpg  (4.7 KB)
  [24/45] ✓ 377b94a0.jpg  (6.5 KB)
  [25/45] ✓ 6e69c887.jpg  (4.5 KB)
  [26/45] ✓ bfb1cc24.jpg  (3.5 KB)
  [27/45] ✓ 3ad61d5d.jpg  (7.6 KB)
  [28/45] ✓ 8

L4 real e fake


In [6]:
# ============================================================
# Download 50 REAL + 50 FAKE images from the OpenFake dataset
# Source : ComplexDataLab/OpenFake on HuggingFace (research use)
# Requires: requests  (pip install requests)
# ============================================================
import os, time, requests

# ── Configuration ────────────────────────────────────────────
PROJECT_DIR  = os.path.dirname(os.path.abspath('__file__'))
REAL_DIR     = os.path.join(PROJECT_DIR, 'openfake_real')
FAKE_DIR     = os.path.join(PROJECT_DIR, 'openfake_fake')
N_EACH       = 50        # images per class
SPLIT        = 'test'    # 'test' | 'train' | 'validation'
CONFIG       = 'core'
PAGE_SIZE    = 100       # rows to fetch per API call
DELAY_S      = 0.1       # polite pause between image downloads

ROWS_API = ('https://datasets-server.huggingface.co/rows'
            '?dataset=ComplexDataLab/OpenFake'
            f'&config={CONFIG}&split={SPLIT}')

os.makedirs(REAL_DIR, exist_ok=True)
os.makedirs(FAKE_DIR, exist_ok=True)

# ── Helpers ──────────────────────────────────────────────────
def fetch_rows(offset, length=PAGE_SIZE):
    url = f'{ROWS_API}&offset={offset}&length={length}'
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json().get('rows', [])

def save_image(src_url, dest_path, idx, total, label):
    resp = requests.get(src_url, timeout=30)
    resp.raise_for_status()
    with open(dest_path, 'wb') as f:
        f.write(resp.content)
    kb = len(resp.content) / 1024
    print(f'  [{idx:02d}/{total}] {label:4s}  ✓  {os.path.basename(dest_path)}  ({kb:.1f} KB)')

# ── Main loop: page through dataset until we have 50+50 ──────
reals_saved, fakes_saved = 0, 0
offset = 0

print(f'Fetching rows from OpenFake ({SPLIT}/{CONFIG}) …\n')

while reals_saved < N_EACH or fakes_saved < N_EACH:
    rows = fetch_rows(offset)
    if not rows:
        print('No more rows available.')
        break

    for rw in rows:
        row   = rw['row']
        label = row.get('label', '').strip().lower()
        img   = row.get('image', {})
        src   = img.get('src', '') if isinstance(img, dict) else ''

        if not src:
            continue

        if label == 'real' and reals_saved < N_EACH:
            idx   = reals_saved + 1
            fname = f'real_{idx:03d}.jpg'
            dest  = os.path.join(REAL_DIR, fname)
            if not os.path.exists(dest):
                try:
                    save_image(src, dest, idx, N_EACH, 'REAL')
                    time.sleep(DELAY_S)
                except Exception as e:
                    print(f'  [REAL {idx}] ERROR: {e}')
                    continue
            else:
                print(f'  [{idx:02d}/{N_EACH}] REAL  – already exists ({fname})')
            reals_saved += 1

        elif label == 'fake' and fakes_saved < N_EACH:
            idx   = fakes_saved + 1
            fname = f'fake_{idx:03d}.jpg'
            dest  = os.path.join(FAKE_DIR, fname)
            if not os.path.exists(dest):
                try:
                    save_image(src, dest, idx, N_EACH, 'FAKE')
                    time.sleep(DELAY_S)
                except Exception as e:
                    print(f'  [FAKE {idx}] ERROR: {e}')
                    continue
            else:
                print(f'  [{idx:02d}/{N_EACH}] FAKE  – already exists ({fname})')
            fakes_saved += 1

        if reals_saved >= N_EACH and fakes_saved >= N_EACH:
            break

    offset += PAGE_SIZE

print(f'\nDone!')
print(f'  REAL images : {reals_saved}  →  {REAL_DIR}')
print(f'  FAKE images : {fakes_saved}  →  {FAKE_DIR}')



Fetching rows from OpenFake (test/core) …

  [01/50] FAKE  ✓  fake_001.jpg  (133.2 KB)
  [01/50] REAL  ✓  real_001.jpg  (15.4 KB)
  [02/50] REAL  ✓  real_002.jpg  (40.1 KB)
  [03/50] REAL  ✓  real_003.jpg  (320.9 KB)
  [04/50] REAL  ✓  real_004.jpg  (249.2 KB)
  [02/50] FAKE  ✓  fake_002.jpg  (132.2 KB)
  [05/50] REAL  ✓  real_005.jpg  (25.5 KB)
  [03/50] FAKE  ✓  fake_003.jpg  (84.5 KB)
  [04/50] FAKE  ✓  fake_004.jpg  (135.5 KB)
  [05/50] FAKE  ✓  fake_005.jpg  (107.2 KB)
  [06/50] REAL  ✓  real_006.jpg  (41.0 KB)
  [07/50] REAL  ✓  real_007.jpg  (37.0 KB)
  [08/50] REAL  ✓  real_008.jpg  (30.6 KB)
  [06/50] FAKE  ✓  fake_006.jpg  (156.9 KB)
  [07/50] FAKE  ✓  fake_007.jpg  (157.0 KB)
  [09/50] REAL  ✓  real_009.jpg  (62.7 KB)
  [10/50] REAL  ✓  real_010.jpg  (42.8 KB)
  [08/50] FAKE  ✓  fake_008.jpg  (780.8 KB)
  [11/50] REAL  ✓  real_011.jpg  (35.0 KB)
  [09/50] FAKE  ✓  fake_009.jpg  (167.1 KB)
  [10/50] FAKE  ✓  fake_010.jpg  (307.9 KB)
  [12/50] REAL  ✓  real_012.jpg  (281.3 KB)

In [1]:
import os
import sys
import csv
import cv2

# --- Configurazione ---
IMAGE_DIR = "'//Users//andrea//Documents//UNI//Computer Vision//Project//images'"
OUTPUT_CSV = "dataset_manifest.csv"
CSV_COLUMNS = ["image_id", "file_path", "is_fake", "sensitivity_level", "is_satire", "source", "notes"]
WINDOW_NAME = "Annotatore Deepfake Sensitivity"
MAX_DISPLAY_W = 1000
MAX_DISPLAY_H = 800


def init_csv(path):
    if not os.path.exists(path):
        with open(path, mode="w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(CSV_COLUMNS)


def load_annotated_ids(path):
    """Dedup su image_id (nome file), non su file_path: resta valido anche
    se lo script viene lanciato da working directory diverse (path relativi
    vs assoluti che altrimenti non matcherebbero più)."""
    annotated = set()
    with open(path, mode="r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            annotated.add(row["image_id"])
    return annotated


def resize_for_display(img, max_w=MAX_DISPLAY_W, max_h=MAX_DISPLAY_H):
    h, w = img.shape[:2]
    scale = min(max_w / w, max_h / h, 1.0)  # non ingrandire mai le immagini piccole
    if scale < 1.0:
        return cv2.resize(img, (int(w * scale), int(h * scale)))
    return img


def window_closed(name):
    """Rileva anche la chiusura della finestra dalla X, non solo l'ESC."""
    try:
        return cv2.getWindowProperty(name, cv2.WND_PROP_VISIBLE) < 1
    except cv2.error:
        return True


def wait_for_key(valid_keys):
    """Blocca finché non viene premuto un tasto valido, ESC, o si chiude la finestra.
    Ritorna il codice del tasto, oppure None se bisogna interrompere il programma."""
    while True:
        key = cv2.waitKey(50) & 0xFF  # poll ogni 50ms: permette di controllare anche la finestra
        if key == 27 or window_closed(WINDOW_NAME):
            return None
        if key in valid_keys:
            return key


def append_row(path, row):
    with open(path, mode="a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(row)


def main():
    if not os.path.isdir(IMAGE_DIR):
        sys.exit(f"Cartella immagini non trovata: {IMAGE_DIR}")

    init_csv(OUTPUT_CSV)
    annotated_ids = load_annotated_ids(OUTPUT_CSV)

    image_files = sorted(
        f for f in os.listdir(IMAGE_DIR)
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
    )
    todo = [f for f in image_files if f not in annotated_ids]

    print("--- CONTROLLI ---")
    print("1. Autenticità: Premi [R] per Real, [F] per Fake")
    print("2. Sensibilità: Premi [0-5]")
    print("3. Satira: Premi [Y] per Sì, [N] per No")
    print("4. Premi [ESC] o chiudi la finestra in qualsiasi momento per uscire\n")
    print(f"Immagini da annotare: {len(todo)} (già annotate: {len(annotated_ids)})\n")

    cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)

    try:
        for i, img_name in enumerate(todo, start=1):
            img_path = os.path.join(IMAGE_DIR, img_name)
            img = cv2.imread(img_path)
            if img is None:
                print(f"[SKIP] Impossibile leggere {img_name}")
                continue

            display_img = resize_for_display(img)
            cv2.imshow(WINDOW_NAME, display_img)
            cv2.setWindowTitle(WINDOW_NAME, f"{WINDOW_NAME} — {i}/{len(todo)}: {img_name}")

            key = wait_for_key({ord('r'), ord('R'), ord('f'), ord('F')})
            if key is None:
                break
            is_fake = 1 if key in (ord('f'), ord('F')) else 0

            key = wait_for_key({ord(str(d)) for d in range(6)})
            if key is None:
                break
            sensitivity = int(chr(key))

            key = wait_for_key({ord('y'), ord('Y'), ord('n'), ord('N')})
            if key is None:
                break
            is_satire = 1 if key in (ord('y'), ord('Y')) else 0

            append_row(OUTPUT_CSV, [img_name, img_path, is_fake, sensitivity, is_satire, "Manual", ""])
            print(f"Salvato ({i}/{len(todo)}): {img_name} -> Fake: {is_fake}, Level: {sensitivity}, Satira: {is_satire}")
    finally:
        cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

SystemExit: Cartella immagini non trovata: '//Users//andrea//Documents//UNI//Computer Vision//Project//images'

/Users/andrea/anaconda3/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3513: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# ============================================================
# Download 50 images from MS-COCO val2017
# No auth needed — images are publicly served at cocodataset.org
# Images cover diverse real-world scenes (people, animals, objects)
# ============================================================
import os, time, requests

PROJECT_DIR = os.path.dirname(os.path.abspath('__file__'))
OUTPUT_DIR  = os.path.join(PROJECT_DIR, 'coco_images')
IMAGE_BASE  = 'http://images.cocodataset.org/val2017/'
DELAY_S     = 0.1
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 50 diverse image IDs sampled from COCO val2017
# (covers people, animals, food, vehicles, indoor/outdoor scenes)
IMAGE_IDS = [
       139,   285,   632,   724,  1000,  1268,  1296,  1353,  1425,  1503,
      2006,  2261,  2413,  3661,  3845,  4134,  4395,  5001,  5477,  5512,
      6040,  6471,  7386,  7816,  8021,  8762,  9400,  9448,  9772, 10363,
     10977, 11051, 11760, 12062, 12670, 13177, 13291, 14439, 14731, 15335,
     16228, 17029, 17714, 18273, 19109, 19786, 20247, 21503, 22935, 23034,
]

total = len(IMAGE_IDS)
saved, skipped = 0, 0
print(f'Downloading {total} COCO val2017 images…\n')

for i, img_id in enumerate(IMAGE_IDS, 1):
    filename  = f'{img_id:012d}.jpg'          # COCO naming convention
    dest_path = os.path.join(OUTPUT_DIR, filename)

    if os.path.exists(dest_path):
        print(f'  [{i:02d}/{total}] already exists – {filename}')
        saved += 1
        continue

    url = IMAGE_BASE + filename
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        with open(dest_path, 'wb') as f:
            f.write(r.content)
        kb = len(r.content) / 1024
        print(f'  [{i:02d}/{total}] ✓ {filename}  ({kb:.0f} KB)')
        saved += 1
    except Exception as e:
        print(f'  [{i:02d}/{total}] ✗ {filename}  ERROR: {e}')
        skipped += 1

    time.sleep(DELAY_S)

print(f'\nDone!  {saved} images saved to: {OUTPUT_DIR}')
if skipped:
    print(f'       {skipped} failed (network issue — re-run to retry)')


In [ ]:
# ============================================================
# Download 50 images from Labeled Faces in the Wild (LFW)
# via Hugging Face Datasets Server API
# No auth needed, lightweight direct image downloads
# ============================================================
import os, time, requests

PROJECT_DIR = os.path.dirname(os.path.abspath('__file__'))
OUTPUT_DIR  = os.path.join(PROJECT_DIR, 'lfw_images')
TOTAL_TARGET = 50
DELAY_S     = 0.1
os.makedirs(OUTPUT_DIR, exist_ok=True)

HEADERS = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)'}

def fetch_lfw_page(offset, length=100):
    url = f"https://datasets-server.huggingface.co/rows?dataset=vilsonrodrigues%2Flfw&config=default&split=train&offset={offset}&length={length}"
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return r.json().get('rows', [])

print(f'Fetching metadata for {TOTAL_TARGET} images from LFW…')
rows = fetch_lfw_page(0, TOTAL_TARGET)

saved, skipped = 0, 0
print(f'\nDownloading images…')

for i, row in enumerate(rows, 1):
    try:
        img_url = row['row']['image']['src']
        label = row['row']['label']
        # Just saving as lfw_001.jpg, etc since original filenames aren't in this API response
        filename = f'lfw_{i:03d}.jpg'
        dest_path = os.path.join(OUTPUT_DIR, filename)

        if os.path.exists(dest_path):
            print(f'  [{i:02d}/{TOTAL_TARGET}] already exists – {filename}')
            saved += 1
            continue

        # Download the image
        r_img = requests.get(img_url, headers=HEADERS, timeout=20)
        r_img.raise_for_status()
        with open(dest_path, 'wb') as f:
            f.write(r_img.content)
        
        kb = len(r_img.content) / 1024
        print(f'  [{i:02d}/{TOTAL_TARGET}] ✓ {filename}  ({kb:.0f} KB) - Label ID: {label}')
        saved += 1
        time.sleep(DELAY_S)
        
    except Exception as e:
        print(f'  [{i:02d}/{TOTAL_TARGET}] ✗ ERROR: {e}')
        skipped += 1

print(f'\nDone!  {saved} images saved to: {OUTPUT_DIR}')
